In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import category_encoders as ce
from sklearn.compose import ColumnTransformer

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, average_precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [ ]:
df = pd.read_csv('../data/employee-attrition.csv')
df.columns = df.columns.str.strip().str.lower()


## Feature Engineering

In [53]:
def create_interaction_features(df):
    """
    Create interaction features that capture relationships between variables.
    """
    df_new = df.copy()
    
    # Age × Experience alignment
    df_new['age_experience_ratio'] = df_new['age'] / (df_new['totalworkingyears'] + 1)
    
    # Income × Job Level alignment
    df_new['income_per_joblevel'] = df_new['monthlyincome'] / (df_new['joblevel'] + 1)
    
    # OverTime × WorkLifeBalance conflict
    df_new['overtime_wlb_conflict'] = ((df_new['overtime'] == 'Yes').astype(int) * 
                                        (5 - df_new['worklifebalance']))
    
    # Distance × BusinessTravel burden
    travel_map = {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
    df_new['commute_burden'] = (df_new['distancefromhome'] * 
                                 df_new['businesstravel'].map(travel_map))
    
    # Job satisfaction × Environment satisfaction
    df_new['overall_job_satisfaction'] = (df_new['jobsatisfaction'] * 
                                           df_new['environmentsatisfaction'])
    
    # Years at company × Job level (career progression)
    df_new['career_progression_rate'] = df_new['joblevel'] / (df_new['yearsatcompany'] + 1)
    
    return df_new

def create_polynomial_features(df, features, degree=2):
    """
    Create polynomial features for key numeric variables.
    """
    df_new = df.copy()
    
    for feature in features:
        if feature in df.columns:
            df_new[f'{feature}_squared'] = df_new[feature] ** 2
            if degree >= 3:
                df_new[f'{feature}_cubed'] = df_new[feature] ** 3
    
    return df_new

def create_binned_features(df):
    """
    Create categorical bins for continuous variables.
    """
    df_new = df.copy()
    
    # Age groups
    df_new['age_group'] = pd.cut(df_new['age'], 
                                   bins=[0, 30, 45, 100], 
                                   labels=['Young', 'Mid-Career', 'Senior'])
    
    # Income quartiles
    df_new['income_quartile'] = pd.qcut(df_new['monthlyincome'], 
                                          q=4, 
                                          labels=['Q1', 'Q2', 'Q3', 'Q4'])
    
    # Tenure groups
    df_new['tenure_group'] = pd.cut(df_new['yearsatcompany'], 
                                      bins=[-1, 2, 5, 10, 100], 
                                      labels=['New', 'Mid', 'Experienced', 'Veteran'])
    
    # Distance categories
    df_new['distance_category'] = pd.cut(df_new['distancefromhome'], 
                                           bins=[-1, 5, 15, 100], 
                                           labels=['Near', 'Medium', 'Far'])
    
    return df_new

def create_ratio_features(df):
    """
    Create ratio features that normalize by relevant denominators.
    """
    df_new = df.copy()
    
    # Income per year of experience
    df_new['income_per_exp_year'] = df_new['monthlyincome'] / (df_new['totalworkingyears'] + 1)
    
    # Promotions per year at company
    df_new['promotion_frequency'] = 1 / (df_new['yearssincelastpromotion'] + 1)
    
    # Training hours per year
    df_new['training_intensity'] = df_new['trainingtimeslastyear'] / (df_new['yearsatcompany'] + 1)
    
    # Time in current role vs total tenure
    df_new['role_stability'] = df_new['yearsincurrentrole'] / (df_new['yearsatcompany'] + 1)
    
    # Manager tenure vs employee tenure
    df_new['manager_stability'] = df_new['yearswithcurrmanager'] / (df_new['yearsatcompany'] + 1)
    
    # Percent salary hike per job level
    df_new['hike_per_level'] = df_new['percentsalaryhike'] / (df_new['joblevel'] + 1)
    
    return df_new

def create_satisfaction_features(df):
    """
    Create composite satisfaction features.
    """
    df_new = df.copy()
    
    satisfaction_cols = ['environmentsatisfaction', 'jobsatisfaction', 
                         'relationshipsatisfaction', 'worklifebalance']
    
    # Average satisfaction
    df_new['avg_satisfaction'] = df_new[satisfaction_cols].mean(axis=1)
    
    # Satisfaction variance (consistency)
    df_new['satisfaction_variance'] = df_new[satisfaction_cols].var(axis=1)
    
    # Minimum satisfaction (weakest link)
    df_new['min_satisfaction'] = df_new[satisfaction_cols].min(axis=1)
    
    # Count of low satisfaction scores (<= 2)
    df_new['low_satisfaction_count'] = (df_new[satisfaction_cols] <= 2).sum(axis=1)
    
    return df_new

def create_tenure_features(df):
    """
    Create advanced tenure-related features.
    """
    df_new = df.copy()
    
    # Time since last promotion relative to tenure
    df_new['promotion_gap_ratio'] = df_new['yearssincelastpromotion'] / (df_new['yearsatcompany'] + 1)
    
    # Role stagnation (years in role - years since promotion)
    df_new['role_stagnation'] = df_new['yearsincurrentrole'] - df_new['yearssincelastpromotion']
    
    # Early career indicator
    df_new['early_career'] = ((df_new['totalworkingyears'] <= 5) & 
                               (df_new['yearsatcompany'] <= 2)).astype(int)
    
    # Job hopper indicator (many companies, low tenure)
    df_new['job_hopper'] = ((df_new['numcompaniesworked'] >= 4) & 
                             (df_new['yearsatcompany'] <= 3)).astype(int)
    
    return df_new

In [54]:
poly_features = ['age', 'totalworkingyears', 'monthlyincome', 'yearsatcompany']
def build_feature_sets(df):

    """
    Create feature sets.
    """

    baseline = df.copy()

    interaction = create_interaction_features(baseline)
    ratio = create_ratio_features(baseline)
    satisfaction = create_satisfaction_features(baseline)
    tenure = create_tenure_features(baseline)
    binned = create_binned_features(baseline)
    polynomial = create_polynomial_features(baseline,poly_features)
    feature_sets = {
        "baseline": baseline,
        "baseline + interaction": interaction,
        "baseline + ratio": ratio,
        "baseline + satisfaction": satisfaction,
        "baseline + tenure": tenure,
        "baseline + polynomial": polynomial,
        "baseline + polynomial + tenure": create_tenure_features(polynomial),
        "baseline + all_engineered": create_binned_features(create_tenure_features(create_satisfaction_features(create_ratio_features(interaction))))
                
    }

    return feature_sets


In [55]:
# Remove constant columns
constant_cols = ['employeecount', 'standardhours', 'over18']
df= df.drop(columns=[col for col in constant_cols if col in df.columns])

In [56]:
attrition_mask = df['attrition'] == 'Yes'
gender_mask = df['gender'] == 'Male'
df['attrition'] = attrition_mask.astype(int)
df['gender'] = gender_mask.astype(int)


In [57]:
x  = df.drop("attrition", axis=1)
y = df['attrition']


In [58]:
results= []
feature_sets = build_feature_sets(x)                        
for name, df_features in feature_sets.items():
    cat_cols = df_features.select_dtypes(include=["object"]).columns
    num_cols = df_features.select_dtypes(include=["int64"]).columns
    preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
        ]
        )

    model = Pipeline([
     ("preprocessing", preprocessor),
     ("classifier", LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
     ])
    scores = cross_val_score(model, df_features, y, cv=5,scoring = "average_precision")

    results.append({'Feature set':name, "Average Precision":scores.mean()})   
results_df =pd.DataFrame(results).sort_values(by="Average Precision",ascending=False)

results_df

,Feature set,Average Precision
5,baseline + polynomial,0.624867
6,baseline + polynomial + tenure,0.620840
4,baseline + tenure,0.615984
0,baseline,0.608443
2,baseline + ratio,0.608443
7,baseline + all_engineered,0.607813
3,baseline + satisfaction,0.605318
1,baseline + interaction,0.601934
